# 05 - IEEE-CIS Rule Explanation Evaluation

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def find_project_root() -> Path | None:
    direct_candidates = [KAGGLE_PROJECT_DIR, Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = sorted(path.parent for path in base.glob("**/configs") if path.is_dir())
            for candidate in matches:
                if (candidate / "src").is_dir():
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and KAGGLE:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
        check=True,
    )
    PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Notebook chỉ đọc frozen IEEE-CIS predictor từ Notebook 02. Predictor probabilities,
calibration và threshold không được train/chọn lại trong bước explanation.

In [ ]:
preflight_manifests = []
preflight_artifacts = []
for root_value in INPUT_ROOTS:
    root = Path(root_value)
    if not root.exists():
        continue
    manifest_paths = [root] if root.is_file() and root.name == "frozen_reference_manifest.json" else list(root.glob("**/frozen_reference_manifest.json"))
    for manifest_path in manifest_paths:
        try:
            manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if str(manifest_payload.get("dataset_name", "")).lower() != "ieee_cis".lower():
            continue
        preflight_manifests.append(manifest_path.resolve())
        artifact_path = manifest_path.parent / str(manifest_payload.get("artifact_file", ""))
        if artifact_path.exists():
            preflight_artifacts.append(artifact_path.resolve())

preflight_table = pd.DataFrame({
    "manifest": [str(path) for path in sorted(set(preflight_manifests))],
})
display(preflight_table)
print("Frozen artifacts:")
for path in sorted(set(preflight_artifacts)):
    print(path)
if not preflight_manifests or not preflight_artifacts:
    raise FileNotFoundError(
        "No complete ieee_cis frozen artifact was found below INPUT_ROOTS. "
        "On Kaggle, attach the corresponding benchmark notebook output; locally, place it below results/runs/notebooks."
    )

In [ ]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact, sha256_file
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "ieee_cis", expected_config=config,
    search_roots=[OUTPUT_BASE / "02_ieee_cis_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
if int(artifact["manifest"]["reference_seed"]) != int(config["evaluation"]["reference_seed"]):
    raise ValueError("Frozen artifact reference seed does not match the locked dataset protocol")
if not QUICK_RUN and str(artifact["manifest"].get("data_source", "")).lower() == "synthetic":
    raise ValueError("Full thesis evaluation cannot consume a synthetic-fallback frozen artifact")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

def write_upstream_lineage(destination, notebook_id, output_files, config_path):
    output_files = list(output_files)
    lineage = {
        "notebook_id": notebook_id,
        "git_commit": GIT_COMMIT,
        "dataset_name": artifact["manifest"]["dataset_name"],
        "data_source": data_source,
        "frozen_data_source": artifact["manifest"].get("data_source"),
        "quick_run": QUICK_RUN,
        "reference_model_key": artifact["manifest"]["model_key"],
        "reference_seed": artifact["manifest"]["reference_seed"],
        "config_sha256": artifact["manifest"]["config_sha256"],
        "audit_source_sha256": audit_pipeline_fingerprint(config_path),
        "frozen_manifest_sha256": sha256_file(artifact["manifest_path"]),
        "frozen_artifact_sha256": sha256_file(artifact["artifact_path"]),
        "output_files": output_files,
        "output_sha256": {
            name: sha256_file(Path(destination) / name) for name in output_files
        },
    }
    lineage_path = Path(destination) / "upstream_lineage.json"
    lineage_path.write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    return lineage_path

In [ ]:
from src.explanation import (
    RuleExplainer, bootstrap_explanation_precision_gain,
    explanation_quality_metrics, rule_quality_table,
)
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "05_ieee_cis_rule_explanation_evaluation"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
explainer = RuleExplainer(engine, config["logic"]["activation_threshold"], config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
explanations["y_true"] = prepared.y_test
display(explanations.head())

## Results

In [ ]:
truth = engine.evaluate(prepared.test_frame)
rule_quality = rule_quality_table(truth, prepared.y_test, config["logic"]["activation_threshold"])
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations, prepared.y_test, probabilities, threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"], seed=config["project"]["seed"],
))
quality_frame = pd.DataFrame([quality])
display(rule_quality.round(4), quality_frame.round(4))

predicted_alert = probabilities >= threshold
cases = explanations.copy()
cases["top_rule_name"] = truth.idxmax(axis=1)
cases["top_rule_strength"] = truth.max(axis=1)
cases["case_type"] = np.select(
    [predicted_alert & (prepared.y_test == 1), predicted_alert & (prepared.y_test == 0),
     (~predicted_alert) & (prepared.y_test == 1)],
    ["true_positive", "false_positive", "false_negative"], default="true_negative",
)
cases["explanation_status"] = np.where(cases["explained"], "explained", "unexplained")
cases["case_group"] = cases["case_type"] + "_" + cases["explanation_status"]
requested_groups = [
    f"{case_type}_{status}"
    for case_type in ("true_positive", "false_positive", "false_negative", "true_negative")
    for status in ("explained", "unexplained")
]
selected_groups = []
for group_name in requested_groups:
    group = cases.loc[cases["case_group"] == group_name]
    if not group.empty:
        selected_groups.append(group.sort_values("predicted_probability", ascending=False).head(2))
selected_cases = (
    pd.concat(selected_groups).copy()
    if selected_groups
    else cases.head(0).copy()
)

fitted_rules = {rule.name: rule for rule in engine.rules}
def serializable_value(value):
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, "item") else value

def audit_evidence(row_index, case_row):
    active_names = list(case_row["rule_names"])
    audit_names = active_names or [case_row["top_rule_name"]]
    strengths = dict(zip(active_names, case_row["rule_strengths"]))
    details = []
    for rule_name in audit_names:
        rule = fitted_rules[rule_name]
        details.append({
            "rule": rule_name,
            "activated": rule_name in active_names,
            "truth_strength": float(strengths.get(rule_name, truth.loc[row_index, rule_name])),
            "conditions": [
                {
                    "feature": condition.feature,
                    "operator": condition.operator,
                    "raw_value": serializable_value(prepared.test_frame.loc[row_index, condition.feature]),
                    "fitted_threshold": condition.threshold,
                    "fitted_missing_value": condition.numeric_fill_value,
                    "softness": condition.softness,
                    "softness_mode": condition.softness_mode,
                }
                for condition in rule.conditions
            ],
        })
    return details

selected_cases["model_decision_threshold"] = threshold
selected_cases["rule_activation_threshold"] = float(config["logic"]["activation_threshold"])
selected_cases["audit_evidence"] = [
    audit_evidence(row_index, row) for row_index, row in selected_cases.iterrows()
]
selected_cases = selected_cases.reset_index(names="row_index")
display(selected_cases[[
    "row_index", "case_group", "y_true", "predicted_probability",
    "model_decision_threshold", "predicted_alert", "explained",
    "top_rule_name", "top_rule_strength", "rule_names", "audit_evidence",
]])
quality_frame.to_csv(output_dir / "ieee_explanation_quality.csv", index=False)
rule_quality.to_csv(output_dir / "ieee_test_rule_quality.csv", index=False)
selected_cases.to_json(
    output_dir / "ieee_explanation_cases.json", orient="records", indent=2, force_ascii=False
)
lineage_path = write_upstream_lineage(
    output_dir,
    "05_IEEE_CIS_Rule_Explanation_Evaluation",
    ["ieee_explanation_quality.csv", "ieee_test_rule_quality.csv", "ieee_explanation_cases.json"],
    PROJECT_ROOT / "configs/ieee_cis.yaml",
)
print({"upstream_lineage": str(lineage_path)})

## Takeaways

In [ ]:
display(Markdown(
    f"- Alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**.\n"
    f"- Explained-alert precision gain: **{quality['explained_alert_precision_gain']:.3f}** "
    f"(95% CI [{quality['precision_gain_ci_low']:.3f}, {quality['precision_gain_ci_high']:.3f}]).\n"
    f"- Unsupported-alert rate: **{quality['unsupported_alert_rate']:.3f}** "
    f"({int(quality['unsupported_alert_count'])}/{int(quality['predicted_alert_count'])} alerts).\n"
    "- Overall consistency is secondary because non-alert rows dominate this imbalanced task. "
    "These are selective rule-evidence diagnostics, not causal or model-faithful explanations."
))